# 3장 1강: 교차표와 카이제곱 독립성 검정 이론 — 실습문제

## 실습 목표

- 두 범주형 변수의 교차표를 구성하고 관측빈도를 해석할 수 있다.
- 행 합계와 열 합계를 이용해 기대빈도를 직접 계산할 수 있다.
- 관측빈도와 기대빈도로 카이제곱 통계량과 자유도를 계산할 수 있다.
- 카이제곱 독립성 검정을 수행하고 기대빈도 조건을 확인할 수 있다.
- p-value와 범주별 비율을 함께 사용하여 변수 간 관련성을 해석할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas, NumPy
- scipy.stats
- `ames_housing.csv`

| 컬럼 | 의미 |
|---|---|
| `OverallQual` | 주택의 전반적인 품질 점수 |
| `YearBuilt` | 건축연도 |
| `CentralAir` | 중앙 냉방시설 유무 |
| `KitchenQual` | 주방 품질 |
| `PavedDrive` | 진입로 포장 상태 |

> 모든 검정의 유의수준은 `α = 0.05`입니다.  
> `chi2_contingency()`에서는 강의자료와 동일하게 `correction=False`를 사용합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.

import numpy as np
import pandas as pd
from scipy import stats

df = pd.read_csv('ames_housing.csv', encoding='utf-8-sig')

print('데이터 크기 :', df.shape)
print('전체 결측치 수 :', df.isnull().sum().sum())
df.head()

데이터 크기 : (1460, 10)
전체 결측치 수 : 0


,SalePrice,GrLivArea,LotArea,OverallQual,KitchenQual,CentralAir,HeatingQC,PavedDrive,Neighborhood,YearBuilt
0,208500,1710,8450,7,Gd,Y,Ex,Y,CollgCr,2003
1,181500,1262,9600,6,TA,Y,Ex,Y,Veenker,1976
2,223500,1786,11250,7,Gd,Y,Ex,Y,CollgCr,2001
3,140000,1717,9550,7,Gd,Y,Gd,Y,Crawfor,1915
4,250000,2198,14260,8,Gd,Y,Ex,Y,NoRidge,2000


---

## 필수 1. 교차표와 카이제곱 통계량 직접 계산

### 문제 1-1. 주택 품질 구간과 중앙 냉방시설의 관계

#### 문제 설명

`OverallQual`을 세 구간으로 나눈 뒤 중앙 냉방시설 유무와의 관계를 확인합니다.

- 낮음: 1~5점
- 보통: 6~7점
- 높음: 8~10점

#### 요구사항

1. `pd.cut()`을 이용해 위 기준으로 `QualityGroup`을 만드세요.
2. 행에는 `QualityGroup`, 열에는 `CentralAir`가 오도록 합계 없는 교차표를 만드세요.
3. `margins=True`인 교차표도 별도로 만들어 행·열 합계를 확인하세요.
4. 합계가 없는 교차표를 NumPy 배열 `observed`로 변환하세요.
5. 행 합계, 열 합계, 전체 합계를 계산하세요.
6. `(행 합계 × 열 합계) / 전체 합계`로 기대빈도를 직접 계산하세요.
7. `Σ(관측-기대)²/기대`로 카이제곱 통계량을 직접 계산하세요.
8. `(행 수-1) × (열 수-1)`로 자유도를 계산하세요.
9. 모든 기대빈도가 5 이상인지 확인하세요.
10. `stats.chi2_contingency(..., correction=False)` 결과와 직접 계산한 값을 비교하세요.
11. p-value를 이용해 두 변수가 관련 있는지 판단하세요.

#### 해석 질문

**Q1.** 교차표 각 칸의 숫자는 무엇을 의미하나요?  
**Q2.** 기대빈도는 어떤 가정 아래 계산되는 값인가요?  
**Q3.** 직접 계산한 카이제곱 통계량과 함수의 결과는 일치하나요?  
**Q4.** 주택 품질 구간과 중앙 냉방시설 유무는 서로 독립이라고 볼 수 있나요?

#### 제출 결과

- 관측빈도 교차표와 주변합
- 기대빈도
- 수동 계산한 카이제곱 통계량과 자유도
- 기대빈도 조건 확인
- 함수 결과와의 비교
- 독립성 판단
- Q1~Q4 답변

In [2]:
# 필수 1 코드를 작성하세요.

# 1. OverallQual을 세 구간으로 나누어 QualityGroup 생성
df['QualityGroup'] = pd.cut(df['OverallQual'], bins=[0, 5, 7, 10], labels=['낮음', '보통', '높음'])


# 2. 합계 없는 교차표
cross_table = pd.crosstab(df['QualityGroup'], df['CentralAir'])

print('[교차표]')
print(cross_table)


# 3. margins=True 합계 교차표
cross_table_margin = pd.crosstab(df['QualityGroup'], df['CentralAir'], margins=True)

print('\n[합계 포함 교차표]')
print(cross_table_margin)


# 4. 교차표를 NumPy 배열로 변환
observed = cross_table.to_numpy()

print('\n[관측빈도]')
print(observed)


# 5. 행 합계, 열 합계, 전체 합계
row_sum = observed.sum(axis=1)
col_sum = observed.sum(axis=0)
total_sum = observed.sum()

print('\n[합계]')
print('행 합계 :', row_sum)
print('열 합계 :', col_sum)
print('전체 합계 :', total_sum)


# 6. 기대빈도 직접 계산
expected = np.outer(row_sum, col_sum) / total_sum

print('\n[기대빈도]')
print(expected.round(3))


# 7. 카이제곱 통계량 직접 계산
chi2_manual = np.sum((observed - expected) ** 2 / expected)

print('\n[직접 계산]')
print(f'카이제곱 통계량 : {chi2_manual:.3f}')


# 8. 자유도 계산
row_num = observed.shape[0]
col_num = observed.shape[1]

dof_manual = (row_num - 1) * (col_num - 1)

print(f'자유도 : {dof_manual}')


# 9. 모든 기대빈도가 5 이상인지 확인
expected_check = np.all(expected >= 5)

print(f'모든 기대빈도가 5 이상인가? : {expected_check}')


# 10. scipy 결과와 비교
chi2, p_value, dof, expected_scipy = stats.chi2_contingency(
    observed,
    correction=False
)

print('\n[카이제곱 결과]')
print(f'카이제곱 통계량 : {chi2:.3f}')
print(f'p-value : {p_value:.3f}')
print('판단 : 유의미한 결과가 도출되었습니다' if p_value <= 0.5 else '판단 : 유의미하지 않은 결과가 도출되었습니다.')
print(f'자유도 : {dof}')
print('기대빈도 :')
print(expected_scipy.round(3))


print('\n[직접 계산과 비교]')
print(f'직접 계산 χ² : {chi2_manual:.3f}')
print(f'Scipy χ²     : {chi2:.3f}')
print(f'직접 계산 자유도 : {dof_manual}')
print(f'Scipy 자유도     : {dof}')

[교차표]
CentralAir     N    Y
QualityGroup         
낮음            71  467
보통            23  670
높음             1  228

[합계 포함 교차표]
CentralAir     N     Y   All
QualityGroup                
낮음            71   467   538
보통            23   670   693
높음             1   228   229
All           95  1365  1460

[관측빈도]
[[ 71 467]
 [ 23 670]
 [  1 228]]

[합계]
행 합계 : [538 693 229]
열 합계 : [  95 1365]
전체 합계 : 1460

[기대빈도]
[[ 35.007 502.993]
 [ 45.092 647.908]
 [ 14.901 214.099]]

[직접 계산]
카이제곱 통계량 : 65.030
자유도 : 2
모든 기대빈도가 5 이상인가? : True

[카이제곱 결과]
카이제곱 통계량 : 65.030
p-value : 0.000
판단 : 유의미한 결과가 도출되었습니다
자유도 : 2
기대빈도 :
[[ 35.007 502.993]
 [ 45.092 647.908]
 [ 14.901 214.099]]

[직접 계산과 비교]
직접 계산 χ² : 65.030
Scipy χ²     : 65.030
직접 계산 자유도 : 2
Scipy 자유도     : 2


### 필수 1 답변 작성란

- **Q1.** 교차표 각 칸의 숫자는 무엇을 의미하나요?  
    - 두 범주형 변수의 해당 조합에 실제로 속한 주택의 개수인 관측빈도

- **Q2.** 기대빈도는 어떤 가정 아래 계산되는 값인가요?  
    - 두 범주형 변수들이 서로 독립적일 것이다.라는 귀무가설이 참일 때 각 칸에 기대되는 빈도

- **Q3.** 직접 계산한 카이제곱 통계량과 함수의 결과는 일치하나요?  
    - 65.03으로 일치

- **Q4.** 주택 품질 구간과 중앙 냉방시설 유무는 서로 독립이라고 볼 수 있나요?
    - 독립이라고 보기 어려움. p-value가 0.05보다 작아 독립이라는 귀무가설을 기각 (서로 연관이 있음)

---

## 필수 2. 카이제곱 독립성 검정과 비율 해석

### 문제 2-1. 건축연도 구간과 중앙 냉방시설의 관계

#### 문제 설명

건축연도를 다음 세 구간으로 나누고 중앙 냉방시설 설치 여부와 관련이 있는지 확인하세요.

- 1980년 이전
- 1980~1999년
- 2000년 이후

#### 요구사항

1. `pd.cut()`로 `YearBuiltGroup`을 만드세요.
2. `YearBuiltGroup`과 `CentralAir`의 교차표를 만드세요.
3. 다음 가설을 작성하세요.
   - H₀: 건축연도 구간과 중앙 냉방시설 유무는 서로 독립이다.
   - H₁: 건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있다.
4. 카이제곱 독립성 검정을 수행하세요.
5. 카이제곱 통계량, p-value, 자유도와 기대빈도를 출력하세요.
6. 기대빈도 중 5 미만인 칸의 개수를 확인하세요.
7. `pd.crosstab(..., normalize="index")`로 건축연도 구간별 냉방시설 비율을 계산하세요.
8. 검정 결과와 행 비율을 함께 이용해 관련성을 해석하세요.

#### 해석 질문

**Q1.** 이 문제는 적합도 검정과 독립성 검정 중 무엇을 사용해야 하나요?  
**Q2.** 자유도는 얼마이며 어떻게 계산되나요?  
**Q3.** 기대빈도 조건은 충족되나요?  
**Q4.** 건축연도 구간과 중앙 냉방시설 유무 사이에는 유의한 관련성이 있나요?  
**Q5.** 카이제곱 검정 결과만으로 건축연도가 냉방시설 설치의 원인이라고 결론 내릴 수 있나요?

#### 제출 결과

- 교차표와 가설
- 카이제곱 검정 결과
- 기대빈도 조건
- 행 비율
- 관련성 및 인과관계 해석
- Q1~Q5 답변

In [3]:
# 필수 2 코드를 작성하세요.

df['YearBuiltGroup'] = pd.cut(df['YearBuilt'], bins=[0, 1979, 1999, 3000], labels=['1980 이전', '1980~1999년', '2000년대 이후'])

# cross_table
cross_table = pd.crosstab(df['YearBuiltGroup'], df['CentralAir'])

print('[교차표]')
print(cross_table)

# 가설을 작성하세요.
#    - H₀: 건축연도 구간과 중앙 냉방시설 유무는 서로 독립이다.
#    - H₁: 건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있다.

chi_statis, chi_pvalue, dof, expect = stats.chi2_contingency(cross_table)

print('\n[카이제곱 검정 결과]')
print('카이제곱 통계량 :', chi_statis)
print('p-value :', chi_pvalue)
print('자유도 :', dof)
print('기대빈도 :\n', expect)


# 기대빈도
expected_df = pd.DataFrame(expect, index=cross_table.index, columns=cross_table.columns)

print('\n[기대빈도]')
print(expected_df)

# 기대빈도 중 5 미만인 칸의 개수
print('\n기대빈도가 5 미만인 칸의 개수 :', len(expect < 5))

# 건축연도 구간별 냉방시설 비율
built_air_raio = pd.crosstab(df['YearBuiltGroup'], df['CentralAir'], normalize='index') * 100

print('건축연도 구간별 냉방시설 비율(%) :')
print(built_air_raio.round(3))



[교차표]
CentralAir       N    Y
YearBuiltGroup         
1980 이전         95  753
1980~1999년       0  224
2000년대 이후        0  388

[카이제곱 검정 결과]
카이제곱 통계량 : 73.33298776695003
p-value : 1.191088511330135e-16
자유도 : 2
기대빈도 :
 [[ 55.17808219 792.82191781]
 [ 14.57534247 209.42465753]
 [ 25.24657534 362.75342466]]

[기대빈도]
CentralAir              N           Y
YearBuiltGroup                       
1980 이전         55.178082  792.821918
1980~1999년      14.575342  209.424658
2000년대 이후       25.246575  362.753425

기대빈도가 5 미만인 칸의 개수 : 3
건축연도 구간별 냉방시설 비율(%) :
CentralAir           N        Y
YearBuiltGroup                 
1980 이전         11.203   88.797
1980~1999년       0.000  100.000
2000년대 이후        0.000  100.000


### 필수 2 답변 작성란

- **Q1.** 이 문제는 적합도 검정과 독립성 검정 중 무엇을 사용해야 하나요?  
    - 건축연도 구간과 냉방시설 유무라는 두 범주형 변수의 관계 확인, 독립성 검정 사용

- **Q2.** 자유도는 얼마이며 어떻게 계산되나요?  
    - 3행 2열 교차표
    - (3-1) * (2-1) = 2
    - 자유도 : 2

- **Q3.** 기대빈도 조건은 충족되나요?  
    - 네. 5 미만인 기대빈도가 0카으로 기대빈도 기준을 만족

- **Q4.** 건축연도 구간과 중앙 냉방시설 유무 사이에는 유의한 관련성이 있나요?  
    -  p-value가 0.05보다 작게 나와 유의미한 결과가 도출되었다.

- **Q5.** 카이제곱 검정 결과만으로 건축연도가 냉방시설 설치의 원인이라고 결론 내릴 수 있나요?
    - 없다. 카이제곱 독립성 검정은 관련성을 확인할 뿐 인과관계를 증명하지 못함.


---

## 과제. 주방 품질과 진입로 포장 상태의 관계

### 문제 3-1. 두 범주형 변수 재분류 후 독립성 검정

#### 문제 설명

기대빈도가 너무 작은 범주를 줄이기 위해 주방 품질과 진입로 포장 상태를 다음과 같이 재분류합니다.

- `KitchenGroup`
  - 우수: `Ex`, `Gd`
  - 보통 이하: `TA`, `Fa`
- `DriveGroup`
  - 완전 포장: `Y`
  - 미포장·부분포장: `N`, `P`

#### 요구사항

1. 위 기준으로 `KitchenGroup`과 `DriveGroup`을 만드세요.
2. 두 변수의 교차표를 작성하세요.
3. 두 변수가 독립이라는 귀무가설과 관련이 있다는 대립가설을 작성하세요.
4. 카이제곱 독립성 검정을 수행하세요.
5. 카이제곱 통계량, p-value, 자유도와 기대빈도를 출력하세요.
6. 모든 기대빈도가 5 이상인지 확인하세요.
7. 주방 품질 집단별 진입로 포장 비율을 계산하세요.
8. 검정 결과와 비율 차이를 함께 사용하여 두 변수의 관련성을 해석하세요.

#### 해석 질문

**Q1.** 이 과제에서 범주를 재분류한 이유는 무엇인가요?  
**Q2.** 기대빈도 조건은 충족되나요?  
**Q3.** 주방 품질 집단과 진입로 포장 상태는 서로 독립이라고 볼 수 있나요?  
**Q4.** 두 주방 품질 집단의 완전 포장 비율은 각각 얼마인가요?

#### 제출 결과

- 재분류 코드와 교차표
- 가설 설정
- 카이제곱 검정 결과
- 기대빈도 조건 확인
- 행 비율과 결과 해석
- Q1~Q4 답변

In [4]:
# 과제 코드를 작성하세요.

# 분류 맵핑
# 주방 품질
kitchen_map = {'Ex' : '우수',
                'Gd' : '우수',
               'TA' :  '보통 이하', 
               'Fa' : '보통 이하'}

df['KitchenGroup'] = df['KitchenQual'].map(kitchen_map)

# 진입로 포장상태
drive_map = {'Y' : '완전 포장',
             'N' : '미포장',
             'P' : '부분포장'}

df['DriveGroup'] = df['PavedDrive'].map(drive_map)

# 교차표
cross_table = pd.crosstab(df['KitchenGroup'], df['DriveGroup'])

print('[교차표]')
print(cross_table)


# 가설 정립
# H₀: 주방품질과 진입로 포장상태는 서로 독립이다.
# H₁: 주방품질과 진입로 포장상태는 서로 관련이 있다.

# 카이제곱 검정
chi_statis, chi_pvalue, dof, expect = stats.chi2_contingency(cross_table)

print('\n[카이제곱 검정 결과]')
print(f'카이제곱 통계량 : {chi_statis:.3f}')
print(f'p-value : {chi_pvalue:.3f}')
print('자유도 :', dof)
print('기대빈도 :\n', expect)

# 기대빈도
expected_df = pd.DataFrame(expect, index=cross_table.index,columns=cross_table.columns)

print('\n[기대빈도]')
display(expected_df.round(3))


# 모든 기대빈도가 5 이상인지 확인
print('\n[기대빈도 조건 확인]')

if (expected >= 5).all():
    print('모든 기대빈도가 5 이상이다.')
else:
    print('기대빈도가 5 미만인 셀이 존재한다.')


# 주방 품질 집단별 진입로 포장 비율
drive_ratio = pd.crosstab(df['KitchenGroup'], df['DriveGroup'],normalize='index') * 100

print('\n[주방 품질 집단별 진입로 포장 비율(%)]')
display(drive_ratio.round(3))


# 검정 결과 + 비율 차이를 이용한 해석
alpha = 0.05

print('\n[결과 해석]')

if chi_pvalue < alpha:
    print(f'p-value({chi_pvalue:.3f}) < 0.05이므로 귀무가설을 기각한다.')
    print('-> 주방 품질 집단과 진입로 포장상태는 서로 관련이 있다고 볼 수 있다.')
else:
    print(f'p-value({chi_pvalue:.3f}) >= 0.05이므로 귀무가설을 기각하지 못한다.')
    print('-> 주방 품질 집단과 진입로 포장상태가 관련이 있다고 볼 통계적 근거가 충분하지 않다.')


[교차표]
DriveGroup    미포장  부분포장  완전 포장
KitchenGroup                  
보통 이하          76    24    674
우수             14     6    666

[카이제곱 검정 결과]
카이제곱 통계량 : 48.431
p-value : 0.000
자유도 : 2
기대빈도 :
 [[ 47.71232877  15.90410959 710.38356164]
 [ 42.28767123  14.09589041 629.61643836]]

[기대빈도]


DriveGroup,미포장,부분포장,완전 포장
KitchenGroup,,,
보통 이하,47.712,15.904,710.384
우수,42.288,14.096,629.616



[기대빈도 조건 확인]
모든 기대빈도가 5 이상이다.

[주방 품질 집단별 진입로 포장 비율(%)]


DriveGroup,미포장,부분포장,완전 포장
KitchenGroup,,,
보통 이하,9.819,3.101,87.080
우수,2.041,0.875,97.085



[결과 해석]
p-value(0.000) < 0.05이므로 귀무가설을 기각한다.
-> 주방 품질 집단과 진입로 포장상태는 서로 관련이 있다고 볼 수 있다.


### 과제 답변 작성란

- **Q1.** 이 과제에서 범주를 재분류한 이유는 무엇인가요? 
    - 기존 범주 중 표본 수가 저은 범주를 유사한 특성끼리 묶어 기대빈도를 충분히 확보하기 위해서이다.

<br>

- **Q2.** 기대빈도 조건은 충족되나요?  
     - 모든 집단에서 기대빈도는 5이상으로 나타났기 때문에 독립성 검정의 기대빈도 조건을 충족된다고 볼 수 있다.

<br>

- **Q3.** 주방 품질 집단과 진입로 포장 상태는 서로 독립이라고 볼 수 있나요?  
    - 카이제곱 검정에서 p-value가 0.05이하로 나타났으므로
    - 독립성이 아닌 서로 관련이 있다고 볼 근거가 있다.

<br>

- **Q4.** 두 주방 품질 집단의 완전 포장 비율은 각각 얼마인가요?
    - 보통 이하 집단의 완전 포장 비율은 약 87%이다.
    - 우수 집단의 완전 포장 비율은 약 97%이다.

---

## 실습 마무리

1. 교차표에서 관측빈도와 기대빈도는 어떻게 다른가요?
    - 관측빈도는 데이터에서 실제로 센 개수, 기대빈도는 두 변수가 독립이라고 가정할 때 주변합으로 예상되는 개수

<br>

2. 기대빈도는 어떤 공식으로 계산하나요?
    - (해당 행 * 해당 열) / 전체합계

<br>

3. 카이제곱 통계량이 커진다는 것은 무엇을 의미하나요?
    - 관측빈도가 독립을 가정한 기대빈도에서 전체적으로 더 크게 벗어난다.

<br>

4. 독립성 검정과 적합도 검정은 변수 개수와 질문에서 어떻게 다른가요?
    |독립성 검정| 적합도 검정|
    |---|---|
    |두 번주형 변수의 관계 확인|하나의 범주형 변수의 관측 분포가 기대한 분포와 일치하는지 확인|

<br>

5. 기대빈도가 5보다 작은 칸이 있다면 무엇을 고려해야 하나요?
    - 기대빈도가 작다면 카이제곱 분포로 근사한 p-value의 정확성이 떨어질 수 있다.